# Model Training and Evaluation
## RecoMart Recommendation System

This notebook implements collaborative filtering using Matrix Factorization (SVD)
to train and evaluate a recommendation model.

Covered in this notebook:
- Data loading and filtering
- Rating construction
- Train/test split
- SVD-based matrix factorization
- Evaluation using RMSE, Precision@K, Recall@K, and NDCG@K
- Simulated MLflow-style tracking



##Imports


In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from collections import defaultdict
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn
import os
import json
from pathlib import Path
import sys

# ... (your existing sys.path and config imports)
sys.path = [
  "/home/abhishek/Documents/Study/dmml_assignment/group_4_dmml",
]
from config import PROCESSED_INTERACTION_DIR

##Load Dataset

In [2]:
# df = pd.read_csv(PROCESSED_INTERACTION_DIR)
folder = Path(PROCESSED_INTERACTION_DIR)
# print(folder)

files = list(folder.glob("interactions_*.csv"))
df = pd.concat(
    (pd.read_csv(file) for file in files),
    ignore_index=True
)


##Filter Valid Interactions :
 This step filters out invalid interactions based on action-level and device-level validation flags. The output summarizes the number and percentage of retained records, ensuring transparency in data quality before model training


In [3]:
valid_df = df[
    (df['action_invalid_data'] == False) &
    (df['device_invalid_data'] == False)
]

valid_df = valid_df[
    valid_df['action_click'] |
    valid_df['action_purchase'] |
    valid_df['action_view']
]

valid_df.shape

valid_df = df[
    (df['action_invalid_data'] == False) &
    (df['device_invalid_data'] == False)
]

print("🔍 Data Filtering Summary")
print("Total interactions:", len(df))
print("Valid interactions:", len(valid_df))
print("Removed interactions:", len(df) - len(valid_df))

valid_ratio = len(valid_df) / len(df) * 100
print(f"Valid data retained: {valid_ratio:.2f}%")

print("\nInvalid data breakdown:")
print("Action-related invalid rows:",
      df['action_invalid_data'].sum())
print("Device-related invalid rows:",
      df['device_invalid_data'].sum())

valid_df.head()



🔍 Data Filtering Summary
Total interactions: 83480
Valid interactions: 83023
Removed interactions: 457
Valid data retained: 99.45%

Invalid data breakdown:
Action-related invalid rows: 237
Device-related invalid rows: 220


,interaction_id,user_id,product_id,rating,event_timestamp,session_duration_sec,action_click,action_invalid_data,action_purchase,action_view,device_invalid_data,device_mobile,device_tablet,event_timestamp_unix
0,INT00016884,U00787,P089,NaN,2025-10-28 01:21:14.342041,0.483565,False,False,False,False,False,False,False,0.168544
1,INT00069689,U01492,P068,NaN,2025-12-14 16:16:21.342041,0.462396,False,False,False,False,False,False,False,0.697686
2,INT00031470,U01242,P083,1.9,2025-11-10 06:03:21.342041,0.315877,False,False,False,True,False,False,False,0.315169
3,INT00048782,U02998,P097,NaN,2025-11-25 18:20:58.342041,0.528134,True,False,False,False,False,False,True,0.487531
4,INT00050861,U01481,P097,NaN,2025-11-27 15:24:40.342041,0.496379,True,False,False,False,False,False,True,0.508394


##Construct Ratings
Since explicit ratings were unavailable, implicit user actions were converted into numerical ratings using weighted aggregation. Purchases were assigned higher weights than clicks and views to reflect stronger user intent.

In [4]:
valid_df['rating'] = (
    5 * valid_df['action_purchase'] +
    3 * valid_df['action_click'] +
    1 * valid_df['action_view']
).astype(float)

valid_df[['user_id', 'product_id', 'rating']].head()


/tmp/ipykernel_59400/1149780979.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_df['rating'] = (


,user_id,product_id,rating
0,U00787,P089,0.0
1,U01492,P068,0.0
2,U01242,P083,1.0
3,U02998,P097,3.0
4,U01481,P097,3.0


##Aggregate User–Item Ratings
Matrix factorization requires exactly one value per user–item pair. Combine all interactions of a user with a product into a single preference score

In [5]:
agg_df = (
    valid_df
    .groupby(['user_id', 'product_id'])['rating']
    .sum()
    .reset_index()
)

agg_df.head()


,user_id,product_id,rating
0,U00001,P002,1.0
1,U00001,P003,3.0
2,U00001,P009,1.0
3,U00001,P014,3.0
4,U00001,P018,2.0


##Encode Users and Items
Encoding converts raw user and product identifiers into contiguous numerical indices so that interactions can be represented as a sparse matrix. This step ensures computational efficiency and is required for matrix factorization models like SVD

In [6]:
# Encode users and items
user_ids = agg_df['user_id'].unique()
item_ids = agg_df['product_id'].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
item_to_index = {p: i for i, p in enumerate(item_ids)}

agg_df['u_idx'] = agg_df['user_id'].map(user_to_index)
agg_df['i_idx'] = agg_df['product_id'].map(item_to_index)

num_users = len(user_to_index)
num_items = len(item_to_index)

print("Encoding Summary")
print("----------------")
print("Total users:", num_users)
print("Total items:", num_items)

print("\nSample user mapping:", list(user_to_index.items())[:5])
print("Sample item mapping:", list(item_to_index.items())[:5])

print("\nEncoded data preview:")
agg_df[['user_id', 'product_id', 'u_idx', 'i_idx']].head()

Encoding Summary
----------------
Total users: 5000
Total items: 100

Sample user mapping: [('U00001', 0), ('U00002', 1), ('U00003', 2), ('U00004', 3), ('U00005', 4)]
Sample item mapping: [('P002', 0), ('P003', 1), ('P009', 2), ('P014', 3), ('P018', 4)]

Encoded data preview:


,user_id,product_id,u_idx,i_idx
0,U00001,P002,0,0
1,U00001,P003,0,1
2,U00001,P009,0,2
3,U00001,P014,0,3
4,U00001,P018,0,4


##Install & Import MLflow

In [7]:

import mlflow


##Train/Test Split (80/20)

In [8]:
np.random.seed(42)

train_mask = np.random.rand(len(agg_df)) < 0.8
train_df = agg_df[train_mask]
test_df = agg_df[~train_mask]

train_df.shape, test_df.shape

np.random.seed(42)
mask = np.random.rand(len(agg_df)) < 0.8

train_df = agg_df[mask]
test_df  = agg_df[~mask]

print("Train/Test Split Summary")
print("------------------------")
print("Total interactions:", len(agg_df))
print("Training interactions:", len(train_df))
print("Testing interactions:", len(test_df))

train_pct = len(train_df) / len(agg_df) * 100
test_pct = len(test_df) / len(agg_df) * 100
print(f"Train split: {train_pct:.2f}%")
print(f"Test split: {test_pct:.2f}%")

print("\nUnique users:")
print("Train:", train_df['user_id'].nunique())
print("Test:", test_df['user_id'].nunique())

print("\nUnique items:")
print("Train:", train_df['product_id'].nunique())
print("Test:", test_df['product_id'].nunique())

print("\nTraining sample:")
train_df.head()

print("\nTesting sample:")
test_df.head()



Train/Test Split Summary
------------------------
Total interactions: 76528
Training interactions: 61300
Testing interactions: 15228
Train split: 80.10%
Test split: 19.90%

Unique users:
Train: 5000
Test: 4777

Unique items:
Train: 100
Test: 100

Training sample:

Testing sample:


,user_id,product_id,rating,u_idx,i_idx
1,U00001,P003,3.0,0,1
7,U00001,P025,0.0,0,7
11,U00001,P060,3.0,0,11
12,U00001,P064,1.0,0,12
33,U00002,P090,1.0,1,29


##Create Sparse User–Item Matrix
We convert interaction data into a sparse matrix to efficiently represent user–item preferences for SVD training

In [9]:
user_item_train = csr_matrix(
    (train_df['rating'],
    (train_df['u_idx'], train_df['i_idx'])),
    shape=(num_users, num_items)
)


print("Sparse User–Item Matrix Created")
print("------------------------------")
print("Matrix shape:", user_item_train.shape)
print("Non-zero entries:", user_item_train.nnz)

density = user_item_train.nnz / (
    user_item_train.shape[0] * user_item_train.shape[1]
)
print(f"Matrix density: {density:.4f}")

print("\nSample (5×5 block):")
user_item_train[:5, :5].toarray()




Sparse User–Item Matrix Created
------------------------------
Matrix shape: (5000, 100)
Non-zero entries: 61300
Matrix density: 0.1226

Sample (5×5 block):


array([[1., 0., 1., 3., 2.],
       [0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 3.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 3.]])

##Train SVD Model

In [10]:

k = 32  # latent factors

U, Sigma, Vt = svds(user_item_train.astype(float), k=k)

from scipy.sparse.linalg import svds

k = 32
U, Sigma, Vt = svds(user_item_train.astype(float), k=k)

print("SVD Model Training Summary")
print("--------------------------")
print("Latent factors (k):", k)
print("U shape:", U.shape)
print("Sigma shape:", Sigma.shape)
print("Vt shape:", Vt.shape)
print("Top 5 singular values:", Sigma[:5])

SVD Model Training Summary
--------------------------
Latent factors (k): 32
U shape: (5000, 32)
Sigma shape: (32,)
Vt shape: (32, 100)
Top 5 singular values: [55.14984651 55.19021732 55.29618099 55.60249697 55.71763298]


##Reconstruct Prediction Matrix
The prediction matrix is reconstructed by multiplying the learned user and item latent factors with their singular values. This matrix estimates preference scores for all user–item pairs, enabling ranking-based recommendation generation.

In [11]:
pred_matrix = U @ np.diag(Sigma) @ Vt
pred_matrix.shape

pred_matrix = U @ np.diag(Sigma) @ Vt

print("Prediction Matrix Reconstructed")
print("-------------------------------")
print("Shape:", pred_matrix.shape)
print("Min score:", pred_matrix.min())
print("Max score:", pred_matrix.max())

print("\nSample prediction block (5×5):")
pd.DataFrame(pred_matrix).iloc[:5, :5]

user_idx = 0
top_k = 5
top_items = np.argsort(pred_matrix[user_idx])[::-1][:top_k]
print("\nTop-5 recommended item indices for User 0:", top_items)

print("Sample:")
pd.DataFrame(pred_matrix).iloc[:5, :5]



Prediction Matrix Reconstructed
-------------------------------
Shape: (5000, 100)
Min score: -2.121122180653189
Max score: 7.289703718458945

Sample prediction block (5×5):

Top-5 recommended item indices for User 0: [16 18 14  6  9]
Sample:


,0,1,2,3,4
0,-0.128072,-0.202886,-0.048303,0.741240,0.960465
1,0.615317,0.021903,0.156166,0.357575,-0.276846
2,0.372380,-0.553200,0.032935,-0.464194,1.627660
3,0.505854,-0.871819,0.022055,-0.131440,0.668376
4,0.458710,0.136715,0.451706,-0.014091,1.478398


##Metric computation: Test data preperation

In [12]:
test_u_idx = test_df['u_idx'].values
test_i_idx = test_df['i_idx'].values
test_r = test_df['rating'].values

test_u_idx = test_df['u_idx'].values
test_i_idx = test_df['i_idx'].values
test_r = test_df['rating'].values

print("Test Data Prepared for Evaluation")
print("-------------------------------")
print("Number of test interactions:", len(test_r))

print("\nSample values:")
print("User indices:", test_u_idx[:5])
print("Item indices:", test_i_idx[:5])
print("Ratings:", test_r[:5])

print("\nIndex validation:")
print("User index range:",
      test_u_idx.min(), "to", test_u_idx.max())
print("Item index range:",
      test_i_idx.min(), "to", test_i_idx.max())



Test Data Prepared for Evaluation
-------------------------------
Number of test interactions: 15228

Sample values:
User indices: [0 0 0 0 1]
Item indices: [ 1  7 11 12 29]
Ratings: [3. 0. 3. 1. 1.]

Index validation:
User index range: 0 to 4999
Item index range: 0 to 99


##RMSE Evaluation

In [13]:
test_u_idx = test_df['u_idx'].values
test_i_idx = test_df['i_idx'].values
test_r = test_df['rating'].values

y_true = test_r
y_pred = pred_matrix[test_u_idx, test_i_idx]

test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("RMSE Evaluation")
print("---------------")
print(f"RMSE on test set: {test_rmse:.4f}")

baseline_pred = np.full_like(y_true, y_true.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_true, baseline_pred))
print(f"Baseline RMSE (mean rating): {baseline_rmse:.4f}")

RMSE Evaluation
---------------
RMSE on test set: 2.1429
Baseline RMSE (mean rating): 1.3792


##Ranking Metrics Setup
This step groups test items per user to enable correct Top-K recommendation evaluation

In [14]:
K = 5 #The model will recommend top 5 items per user

test_user_items = defaultdict(set)
for row in test_df.itertuples():
    test_user_items[row.u_idx].add(row.i_idx)

print("Test User–Item Mapping Created")
print("------------------------------")
print("Number of users in test set:", len(test_user_items))

print("\nSample test interactions per user:")
for u in list(test_user_items.keys())[:3]:
    print(f"User {u} → items {test_user_items[u]}")

items_per_user = [len(v) for v in test_user_items.values()]
print("\nTest item statistics:")
print("Average items per user:", np.mean(items_per_user))
print("Min items per user:", np.min(items_per_user))
print("Max items per user:", np.max(items_per_user))

Test User–Item Mapping Created
------------------------------
Number of users in test set: 4777

Sample test interactions per user:
User 0 → items {1, 11, 12, 7}
User 1 → items {18, 29, 30}
User 2 → items {38, 39, 17, 18, 26}

Test item statistics:
Average items per user: 3.187774754029726
Min items per user: 1
Max items per user: 11


##Precision@K, Recall@K, NDCG@K
Precision@K  - How many recommended items are relevant
Recall@K     - How many relevant items were retrieved
NDCG@K       - Ranking quality (order matters)



In [15]:
precisions, recalls, ndcgs = [], [], []

for u_idx, rel_items in test_user_items.items():
    num_rel = len(rel_items)
    if num_rel == 0:
        continue

    scores = pred_matrix[u_idx, :]
    ranked_items = np.argsort(scores)[::-1]

    top_k = set(ranked_items[:K])
    hits = len(top_k & rel_items)

    precisions.append(hits / K)
    recalls.append(hits / num_rel)

    dcg = sum(
        1 / np.log2(i + 2)
        for i in range(K)
        if ranked_items[i] in rel_items
    )

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(min(K, num_rel))
    )

    ndcgs.append(dcg / idcg if idcg > 0 else 0)





## Display Ranking Metrics

In [16]:
avg_prec = np.mean(precisions)
avg_recall = np.mean(recalls)
avg_ndcg = np.mean(ndcgs)

print(f"Average Precision@{K}: {avg_prec:.4f}")
print(f"Average Recall@{K}: {avg_recall:.4f}")
print(f"Average NDCG@{K}: {avg_ndcg:.4f}")


Average Precision@5: 0.0080
Average Recall@5: 0.0123
Average NDCG@5: 0.0080


## MLflow Tracking and Model Registration

In [17]:
# MLflow Configuration
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("RecoMart-SVD-Model")

# Ensure K is defined for metric naming
K = 5

# Create artifacts directory
artifacts_dir = Path("model_artifacts")
artifacts_dir.mkdir(exist_ok=True)

# Save Model Matrices
u_path = artifacts_dir / "U_matrix.npy"
sigma_path = artifacts_dir / "Sigma.npy"
vt_path = artifacts_dir / "Vt_matrix.npy"

np.save(str(u_path), U)
np.save(str(sigma_path), Sigma)
np.save(str(vt_path), Vt)

# Save model configuration as JSON for reference
model_config = {
    "model_type": "SVD_Collaborative_Filtering",
    "latent_factors_k": int(k),
    "num_users": int(num_users),
    "num_items": int(num_items),
    "user_to_index_mapping": {str(k): v for k, v in list(user_to_index.items())[:10]},
    "item_to_index_mapping": {str(k): v for k, v in list(item_to_index.items())[:10]}
}

config_path = artifacts_dir / "model_config.json"
with open(str(config_path), "w") as f:
    json.dump(model_config, f, indent=4)

# Start MLflow Run
with mlflow.start_run(run_name="svd_cf_training"):
    # Define Parameters
    params = {
        "model_type": "Collaborative Filtering (SVD)",
        "latent_factors_k": int(k),
        "num_users": int(num_users),
        "num_items": int(num_items),
        "rating_construction": "purchase=5, click=3, view=1",
        "train_test_split": "80-20",
        "evaluation_k": int(K)
    }
    
    # Define Metrics
    metrics = {
        "test_rmse": float(test_rmse),
        f"precision_at_{K}": float(avg_prec),
        f"recall_at_{K}": float(avg_recall),
        f"ndcg_at_{K}": float(avg_ndcg)
    }
    
    # Log Parameters
    mlflow.log_params(params)
    
    # Log Metrics
    mlflow.log_metrics(metrics)
    
    # Log Tags
    mlflow.set_tags({
        "team": "Group_4",
        "task": "recommendation_system",
        "model_type": "SVD",
        "dataset": "RecoMart"
    })
    
    # Log Artifacts while run is still active
    mlflow.log_artifacts(str(artifacts_dir), artifact_path="model_matrices")
    
    print("✅ MLflow Run Started")
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print("\nParameters logged:")
    for key, val in params.items():
        print(f"  {key}: {val}")
    print("\nMetrics logged:")
    for key, val in metrics.items():
        print(f"  {key}: {val:.4f}")
    
    print("\n✅ Model artifacts saved and logged to MLflow:")
    print(f"  - U matrix: {u_path}")
    print(f"  - Sigma: {sigma_path}")
    print(f"  - Vt matrix: {vt_path}")
    print(f"  - Model config: {config_path}")
    print(f"\n✅ Artifacts successfully logged to MLflow")
    print(f"View results at: {mlflow.get_tracking_uri()}")

2026/01/25 13:32:51 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: No module named 'git'


✅ MLflow Run Started
Tracking URI: http://127.0.0.1:5000/
Run ID: 0262e836c4a04903a58dd92353d88a7e

Parameters logged:
  model_type: Collaborative Filtering (SVD)
  latent_factors_k: 32
  num_users: 5000
  num_items: 100
  rating_construction: purchase=5, click=3, view=1
  train_test_split: 80-20
  evaluation_k: 5

Metrics logged:
  test_rmse: 2.1429
  precision_at_5: 0.0080
  recall_at_5: 0.0123
  ndcg_at_5: 0.0080

✅ Model artifacts saved and logged to MLflow:
  - U matrix: model_artifacts/U_matrix.npy
  - Sigma: model_artifacts/Sigma.npy
  - Vt matrix: model_artifacts/Vt_matrix.npy
  - Model config: model_artifacts/model_config.json

✅ Artifacts successfully logged to MLflow
View results at: http://127.0.0.1:5000/
🏃 View run svd_cf_training at: http://127.0.0.1:5000/#/experiments/1/runs/0262e836c4a04903a58dd92353d88a7e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Save and Log Model Artifacts to MLflow

In [ ]:
# All model artifacts and MLflow logging is now handled in the previous cell
# This ensures artifacts are logged while the MLflow run context is still active

print("✅ Model training and MLflow logging complete!")
print("All artifacts have been saved and logged to MLflow.")

✅ Model training and MLflow logging complete!
All artifacts have been saved and logged to MLflow.


: 